# MPCount â€” Colab Setup

**Run cells 1-4 once at the start of every session.** After that, jump straight to Training / Test / Inference.

> Warning: Make sure the runtime is set to GPU: Runtime > Change runtime type > T4 GPU

## Cell 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 - Clone or update the code from GitHub

**First time only**: set FIRST_TIME = True to clone the repo.

**Every session after that**: set FIRST_TIME = False to just pull the latest changes.

Fill in your GitHub username and repo name below.

In [ ]:
import os

GITHUB_USERNAME = "gabrielxmit10"   # <-- change this
REPO_NAME       = "MPCount_test1"   # <-- change this

# If your repository is PRIVATE, you must provide a Personal Access Token (PAT).
# Generate one at: https://github.com/settings/tokens (classic token with "repo" scope)
# If your repo is public, leave this as ""
GITHUB_TOKEN    = ""

if GITHUB_TOKEN:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
else:
    REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# Where to keep the code on Drive (persists across sessions)
DRIVE_CODE_DIR  = f"/content/drive/MyDrive/{REPO_NAME}"

FIRST_TIME = True   # <-- set to False after the first clone

if FIRST_TIME:
    !git clone {REPO_URL} {DRIVE_CODE_DIR}
else:
    !git -C {DRIVE_CODE_DIR} pull

# Make this the working directory for the rest of the session
os.chdir(DRIVE_CODE_DIR)
print(f"Working directory: {os.getcwd()}")


## Cell 3 - Install dependencies

Torch and torchvision are already installed by Colab with CUDA support.
We use requirements_colab.txt which skips them to avoid overwriting with a CPU-only version.

In [ ]:
!pip install -r requirements_colab.txt -q
print('Dependencies installed.')

## Cell 4 - Link data from Drive

Your dataset folders (sta, stb, etc.) should live on Drive so they persist.
This cell creates a data/ symlink inside the code folder pointing to them.

Expected Drive layout:
```
MyDrive/
  MPCount_data/
    sta/   <- ShanghaiTech Part A
    stb/   <- ShanghaiTech Part B
```
Adjust DRIVE_DATA_DIR below if your folder is named differently.

In [ ]:
import os
import shutil

# Google Drive does not support Linux symlinks (Errno 95).
# So we copy the data & logs folders directly into the working directory if missing.

BASE_DRIVE_DIR = "/content/drive/MyDrive/MPCount_test1_data"
DRIVE_DATA_SRC = os.path.join(BASE_DRIVE_DIR, "data")
DRIVE_LOGS_SRC = os.path.join(BASE_DRIVE_DIR, "logs")

# Copy data/ if not present
if not os.path.exists("data") and os.path.exists(DRIVE_DATA_SRC):
    print("Copying data/ from Drive (one-time operation)...")
    shutil.copytree(DRIVE_DATA_SRC, "data")
    print("data/ ready.")
elif os.path.exists("data"):
    print("data/ folder already present.")

# Copy logs/ if not present
if not os.path.exists("logs") and os.path.exists(DRIVE_LOGS_SRC):
    print("Copying logs/ from Drive (one-time operation)...")
    shutil.copytree(DRIVE_LOGS_SRC, "logs")
    print("logs/ ready.")
elif os.path.exists("logs"):
    print("logs/ folder already present.")

# Sanity check
if os.path.exists("data"):
    print("\nContents of data/:", os.listdir("data"))
if os.path.exists("logs"):
    print("Contents of logs/:", os.listdir("logs"))


## Cell 5 - Download Pretrained Weights

Run this once to download the deterministic STA pretrained weights from the original authors directly into your weights folder.

In [ ]:
import os

# The weights file should already be in your Drive at:
# MyDrive/MPCount_test1/logs/sta/sta_deterministic.pth
# (Upload it there manually once - it persists forever)
WEIGHTS_FILE = os.path.join(os.getcwd(), "logs", "sta", "sta_deterministic.pth")

if os.path.exists(WEIGHTS_FILE):
    print(f"Weights found: {WEIGHTS_FILE}")
else:
    print(f"ERROR: Weights not found at: {WEIGHTS_FILE}")
    print("Make sure you uploaded the .pth file to MyDrive/MPCount_test1/logs/sta/")


## Cell 6 - Verify GPU

Quick sanity check before starting any training.

In [ ]:
import torch
print('Torch version :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
    print('VRAM          :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

---
## Training

In [ ]:
# Train on ShanghaiTech Part A
!python main.py --config configs/sta_train_fixed.yml --task train

In [ ]:
# Train on ShanghaiTech Part B
!python main.py --config configs/stb_train.yml --task train

## Testing

In [ ]:
# Test: trained on STA, test on STB
!python main.py --config configs/sta_test_stb.yml --task test

In [ ]:
# Test: trained on STB, test on STA
!python main.py --config configs/stb_test_sta.yml --task test

## Inference on a single image or folder

In [ ]:
IMG_PATH   = '/content/drive/MyDrive/MPCount_test1/data/sta/test/IMG_1.jpg'  # or a folder path
# MODEL_PATH = '/content/drive/MyDrive/MPCount_test1/logs/sta/sta_deterministic.pth'                          # adjust to your checkpoint
MODEL_PATH = 'logs/sta/sta_deterministic.pth'                          # adjust to your checkpoint
VIS_DIR    = 'vis_output'

!python inference.py \
    --img_path   {IMG_PATH} \
    --model_path {MODEL_PATH} \
    --vis_dir    {VIS_DIR} \
    --device     cuda